# 03 — Calcular a distância total até o pronto-atendimento mais acessível

**Pergunta:** qual pronto-atendimento exige o menor percurso estimado desde cada setor, usando as ruas?

Diferentemente do notebook antigo de correção, este notebook recalcula todos os setores das 26 capitais, nas versões SUS e Todos. A rede de ruas já baixada é uma entrada; pontos, ligações, rota mínima, grupos e conferências são feitos aqui.

A regra adotada é minimizar a **soma completa**: ligação do setor à rua + percurso na rede + ligação do pronto-atendimento à rua. Assim, a unidade escolhida é a que produz o menor percurso total estimado. Compararemos esse resultado ao cálculo anterior para medir o que mudou.

In [1]:
from pathlib import Path
import gc
import sys
import unicodedata

import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
from IPython.display import display, Markdown
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import dijkstra

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/"PLANO.md").exists())
BASE = RAIZ/"dados/processados/retomada/base"
SAIDA = RAIZ/"dados/processados/retomada"
DIST = SAIDA/"distancias"
DIST.mkdir(parents=True,exist_ok=True)
capitais = pd.read_csv(BASE/"capitais_resumo.csv",dtype={"co_ibge7":str})
atributos = pd.read_csv(BASE/"setores_capitais.csv",dtype={"CD_SETOR":str,"CD_MUN":str})
unidades = pd.read_csv(BASE/"pa_capitais_corrigido.csv",dtype={"CO_CNES":str})
assert not atributos.CD_SETOR.duplicated().any()
def slug(nome):
    return unicodedata.normalize("NFKD",nome).encode("ascii","ignore").decode().split(" - ")[0].lower().replace(" ","_")
def valido(d):
    return (d.populacao>0)&(d.ligacao_m<=2000)&np.isfinite(d.d_m)
print("Capitais com unidades:",int((capitais.n_pa>0).sum()))


Capitais com unidades: 26


## 1. Um código de setor, uma geometria e um total de população

A malha pode trazer várias partes para o mesmo código. Reunimos essas partes antes de criar o ponto representativo. A união conserva os pedaços separados: não preenche o espaço entre eles.

Guardamos a **área oficial uma vez** e ligamos os atributos por código. Isso impede repetir todos os moradores para cada parte do mesmo setor. Os setores sem moradores continuam no arquivo completo, mas não entram nos resumos analíticos.


In [2]:
def preparar_geometrias(capital):
    uf = capital.municipio.split(" - ")[1]
    partes = gpd.read_file(
        RAIZ/"dados/ibge/setores_gpkg"/f"{uf}_setores_CD2022.gpkg",
        where=f"CD_MUN='{capital.co_ibge7}'",columns=["CD_SETOR","CD_SIT","AREA_KM2"]
    )
    partes = partes.loc[partes.CD_SIT!="9"].to_crs(4674)
    crs_m = partes.estimate_utm_crs()
    partes = partes.to_crs(crs_m)
    limite_municipal = partes.geometry.union_all()
    repetidos = partes.CD_SETOR.duplicated(keep=False)
    totais = partes.groupby("CD_SETOR").agg(
        AREA_KM2=("AREA_KM2","first"),n_partes_original=("CD_SETOR","size")
    )
    assert (partes.groupby("CD_SETOR").AREA_KM2.nunique(dropna=False)==1).all()
    simples = partes.loc[~repetidos,["CD_SETOR","geometry"]]
    reunidos = partes.loc[repetidos,["CD_SETOR","geometry"]].dissolve(by="CD_SETOR",as_index=False)
    geo = gpd.GeoDataFrame(pd.concat([simples,reunidos],ignore_index=True),crs=crs_m)
    geo = geo.merge(totais,on="CD_SETOR",validate="one_to_one")
    geo = geo.merge(atributos[["CD_SETOR","populacao","densidade","renda_media","favela"]],
                    on="CD_SETOR",how="left",validate="one_to_one")
    assert not geo.CD_SETOR.duplicated().any()
    assert not geo.geometry.is_empty.any()
    geo["geometria_unida"] = geo.n_partes_original>1
    return geo,limite_municipal,crs_m


## 2. Usar a rede dirigida e montar a matriz invertida

A rede mantém os comprimentos e os sentidos das ruas. Primeiro recortamos a entrada já coletada à faixa de 5 km além da capital e selecionamos a maior componente fortemente conexa: a parte em que os nós conseguem alcançar uns aos outros.

Depois projetamos as coordenadas em metros. Inverter a matriz permite calcular, a partir dos atendimentos, a distância que vai **do setor até eles**, como no exemplo pequeno.

Quando existem duas arestas entre o mesmo par de nós, usamos a menor. O tratamento histórico de comprimentos zero (mínimo 0,01 m) é preservado; não escolhemos outro peso ou critério.


In [3]:
def preparar_rede(chave,limite_municipal,crs):
    arquivo = RAIZ/"dados/osm"/f"{chave}_drive.graphml"
    if not arquivo.exists():
        raise FileNotFoundError(f"Falta a rede coletada: {arquivo}")
    rede = ox.load_graphml(arquivo)
    limite = gpd.GeoSeries([limite_municipal.buffer(5000)],crs=crs).to_crs(4326).iloc[0]
    rede = ox.truncate.truncate_graph_polygon(rede,limite)
    rede = ox.truncate.largest_component(rede,strongly=True)
    rede = ox.project_graph(rede,to_crs=crs)
    return rede

def matriz_invertida(rede):
    nos = list(rede.nodes)
    indice = {n:i for i,n in enumerate(nos)}
    arestas = pd.DataFrame(
        [(indice[v],indice[u],d["length"]) for u,v,d in rede.edges(data=True)],
        columns=["de","para","comprimento"]
    )
    arestas = arestas.groupby(["de","para"],as_index=False).comprimento.min()
    arestas["comprimento"] = arestas.comprimento.clip(lower=.01)
    matriz = coo_matrix((arestas.comprimento,(arestas.de,arestas.para)),
                        shape=(len(nos),len(nos))).tocsr()
    return matriz,indice


## 3. Ligar os pontos à rede e registrar unidades que compartilham nó

Cada setor recebe um ponto dentro da sua geometria. Ligamos esse ponto ao nó de rua mais próximo. Para cada unidade, também registramos o nó mais próximo e a distância dessa ligação.

Se duas unidades estiverem ligadas ao mesmo nó, elas continuam sendo duas unidades na contagem de oferta, mas formam uma área de atendimento compartilhada. Para calcular a distância mínima, o algoritmo considera cada unidade e seu próprio custo de ligação à rua. O nome usado como representante do grupo é o da unidade com a menor ligação até aquele nó.

In [4]:
def ligar_atendimentos(rede,capital,versao,crs):
    pa = unidades.loc[(unidades.municipio==capital)&(unidades.atende_sus | (versao=="Todos"))].copy()
    pa = gpd.GeoDataFrame(pa,geometry=gpd.points_from_xy(pa.lon,pa.lat),crs=4326).to_crs(crs)
    pa = pa.reset_index(drop=True)
    pa["no"],pa["ligacao_m"] = ox.distance.nearest_nodes(
        rede,pa.geometry.x,pa.geometry.y,return_dist=True
    )
    fontes = pa.sort_values("ligacao_m").drop_duplicates("no").reset_index(drop=True)
    grupos = []
    for f in fontes.itertuples():
        membros = pa.loc[pa.no==f.no]
        grupos.append({
            "capital":capital,"versao":versao,"CO_CNES_representante":f.CO_CNES,
            "no":str(int(f.no)),"n_unidades":len(membros),
            "codigos_unidades":";".join(membros.CO_CNES),
            "nomes_unidades":" | ".join(membros.NO_FANTASIA),
        })
    return pa,fontes,grupos


In [5]:
def calcular_distancias(geo,pontos,nos_setores,ligacoes,matriz,indice,pontos_pa):
    # Acrescentamos uma fonte auxiliar para cada unidade. O custo da aresta
    # fonte -> nó da rua é a ligação daquela unidade à rede.
    n = matriz.shape[0]
    original = matriz.tocoo()
    fontes_aux = np.arange(n, n + len(pontos_pa))
    nos_pa = np.array([indice[no] for no in pontos_pa.no])
    matriz_total = coo_matrix((
        np.r_[original.data, pontos_pa.ligacao_m.to_numpy()],
        (np.r_[original.row, fontes_aux], np.r_[original.col, nos_pa])
    ), shape=(n + len(pontos_pa), n + len(pontos_pa))).tocsr()

    # O grafo está invertido para medir do setor até o serviço.
    # min_only=True mantém, para cada nó, a distância e a unidade vencedora.
    dist_total, _, origem = dijkstra(
        matriz_total, directed=True, indices=fontes_aux,
        min_only=True, return_predecessors=True
    )
    indices_setores = np.array([indice[n] for n in nos_setores])
    posicao_unidade = {int(n_aux): i for i, n_aux in enumerate(fontes_aux)}
    selecao = [posicao_unidade.get(int(o)) for o in origem[indices_setores]]
    assert all(i is not None for i in selecao)
    escolhidas = pontos_pa.iloc[selecao].reset_index(drop=True)

    d = pd.DataFrame(geo.drop(columns="geometry"))
    d["CO_CNES"] = escolhidas.CO_CNES.to_numpy()
    d["ligacao_m"] = ligacoes
    d["ligacao_pa_m"] = escolhidas.ligacao_m.to_numpy()
    d["dist_rede_m"] = dist_total[indices_setores] - d.ligacao_pa_m.to_numpy()
    d["d_m"] = dist_total[indices_setores] + d.ligacao_m.to_numpy()
    d["euclid_m"] = [p.distance(q) for p,q in zip(pontos,escolhidas.geometry)]
    assert np.isfinite(d.d_m).all()
    assert (d.d_m >= d.euclid_m - 1).all(), "Distância de rede menor que linha reta: investigar."
    return d,indices_setores

## 4. Como o Dijkstra minimiza a soma completa

A conta desejada para cada setor é: **ligação setor–rua + rota pelas ruas + ligação rua–unidade**. A primeira parcela é igual qualquer que seja o destino; as outras duas podem variar entre unidades.

Para representar isso no grafo, criamos uma fonte auxiliar para cada unidade e uma aresta dessa fonte até o nó de rua com o comprimento da ligação da unidade. O Dijkstra parte de todas essas fontes ao mesmo tempo. Para cada setor, ele devolve a menor soma e qual unidade a alcança com esse menor custo. Depois acrescentamos a ligação do setor à rua.

Esse procedimento inclui a ligação da unidade antes de escolher o destino. Também funciona se várias unidades compartilham um nó: cada uma entra com seu próprio custo, e ambas continuam contadas na densidade de estabelecimentos.

In [6]:
# Exemplo didático: a menor rota entre nós não precisa dar a menor soma total.
# Unidade A: 10 m de rua + 100 m de ligação; unidade B: 12 m de rua + 1 m de ligação.
# O método completo deve escolher B (13 m), não A (110 m).
# A matriz invertida guarda o percurso dos nós das unidades em direção ao setor 0.
exemplo = coo_matrix((
    [10., 12., 100., 1.],
    ([1, 2, 3, 4], [0, 0, 1, 2])
), shape=(5, 5)).tocsr()
dist, _, origem = dijkstra(exemplo, directed=True, indices=[3, 4], min_only=True, return_predecessors=True)
assert dist[0] == 13
assert origem[0] == 4
print("Conferência do exemplo: a menor soma total escolhe B: 12 m de rua + 1 m de ligação = 13 m.")

Conferência do exemplo: a menor soma total escolhe B: 12 m de rua + 1 m de ligação = 13 m.


## 5. Recalcular as 26 capitais e comparar com a regra anterior

Para cada capital e versão, comparamos os novos resultados aos arquivos anteriores por código de setor. Essa comparação serve para medir a mudança de regra, não para exigir que o método novo reproduza o antigo.

Conferimos população, área e atributos por código; uma linha por setor; distâncias finitas; distância de rede não menor que a linha reta; e se a distância total nova nunca supera a distância anterior para o mesmo setor. Também registramos setores cuja distância caiu mais de 1 m e moradores nesses setores.

In [7]:
grupos_todos,resumos,comparacoes = [],[],[]
for capital in capitais.loc[capitais.n_pa>0].itertuples():
    chave = slug(capital.municipio)
    geo,limite,crs = preparar_geometrias(capital)
    rede = preparar_rede(chave,limite,crs)
    matriz,indice = matriz_invertida(rede)
    pontos = geo.geometry.representative_point()
    nos_setores,ligacoes = ox.distance.nearest_nodes(rede,pontos.x,pontos.y,return_dist=True)
    for versao,sufixo in [("SUS","drive_sus"),("Todos","drive")]:
        pontos_pa,fontes,grupos = ligar_atendimentos(rede,capital.municipio,versao,crs)
        d,indices_setores = calcular_distancias(geo,pontos,nos_setores,ligacoes,matriz,indice,pontos_pa)
        d["capital"] = capital.municipio
        anterior = pd.read_csv(
            RAIZ/"dados/processados/corrigidos/distancias"/f"{chave}_{sufixo}_setores.csv",
            dtype={"CD_SETOR":str,"CO_CNES":str}
        ).set_index("CD_SETOR").sort_index()
        atual = d.set_index("CD_SETOR").sort_index()
        assert atual.index.equals(anterior.index)
        for coluna in ["populacao","AREA_KM2","densidade","renda_media","ligacao_m"]:
            np.testing.assert_allclose(atual[coluna],anterior[coluna],atol=1e-5,rtol=1e-10,equal_nan=True,
                                       err_msg=f"{capital.municipio}, {versao}, {coluna}")
        ganho = anterior.d_m - atual.d_m
        assert (ganho >= -1e-5).all(), "A menor soma total não pode piorar em relação à regra anterior."
        mudou = atual.CO_CNES != anterior.CO_CNES
        validos_antes = (anterior.populacao > 0) & (anterior.ligacao_m <= 2000) & np.isfinite(anterior.d_m)
        mudou_dist = validos_antes & (ganho > 1)
        comparacoes.append({
            "capital":capital.municipio,"versao":versao,
            "setores_validos":int(validos_antes.sum()),
            "setores_com_unidade_diferente":int((validos_antes & mudou).sum()),
            "setores_com_reducao_maior_1m":int(mudou_dist.sum()),
            "moradores_em_setores_com_reducao_maior_1m":int(anterior.loc[mudou_dist,"populacao"].sum()),
            "reducao_media_m":float(ganho[validos_antes].mean()),
            "reducao_maxima_m":float(ganho[validos_antes].max()),
        })
        d.to_csv(DIST/f"{chave}_{sufixo}_setores.csv",index=False)
        grupos_todos.extend(grupos)
        validos = d.loc[valido(d)]
        resumos.append({
            "capital":capital.municipio,"versao":versao,"setores":len(d),
            "setores_validos":len(validos),"populacao_valida":validos.populacao.sum(),
            "nos_rede":rede.number_of_nodes(),"arestas_rede":rede.number_of_edges(),
            "setores_unidos":int((d.n_partes_original>1).sum()),
            "grupos":len(grupos),"estabelecimentos":sum(g["n_unidades"] for g in grupos),
            "maior_ligacao_valida_m":validos.ligacao_m.max(),
        })
    pd.DataFrame(resumos).to_csv(SAIDA/"resumo_distancias_completas.csv",index=False)
    pd.DataFrame(comparacoes).to_csv(SAIDA/"comparacao_regra_anterior.csv",index=False)
    print(capital.municipio,": todos os setores recalculados e conferidos, SUS e todos.",flush=True)
    del rede,matriz,geo,pontos
    gc.collect()

grupos = pd.DataFrame(grupos_todos)
grupos.to_csv(SAIDA/"grupos_unidades.csv",index=False)
comparacoes = pd.DataFrame(comparacoes)
comparacoes.to_csv(SAIDA/"comparacao_regra_anterior.csv",index=False)
resumos = pd.DataFrame(resumos)
display(resumos.groupby("versao")[["setores","setores_validos","populacao_valida","grupos","estabelecimentos"]].sum())
display(comparacoes.groupby("versao").agg(
    setores_com_unidade_diferente=("setores_com_unidade_diferente","sum"),
    setores_com_reducao_maior_1m=("setores_com_reducao_maior_1m","sum"),
    moradores_em_setores_com_reducao_maior_1m=("moradores_em_setores_com_reducao_maior_1m","sum"),
    reducao_maxima_m=("reducao_maxima_m","max"),
).round(2))


São Paulo - SP : todos os setores recalculados e conferidos, SUS e todos.


Rio de Janeiro - RJ : todos os setores recalculados e conferidos, SUS e todos.


Brasília - DF : todos os setores recalculados e conferidos, SUS e todos.


Fortaleza - CE : todos os setores recalculados e conferidos, SUS e todos.


Salvador - BA : todos os setores recalculados e conferidos, SUS e todos.


Belo Horizonte - MG : todos os setores recalculados e conferidos, SUS e todos.


... (18 outputs left out)

Vitória - ES : todos os setores recalculados e conferidos, SUS e todos.


Palmas - TO : todos os setores recalculados e conferidos, SUS e todos.


,setores,setores_validos,populacao_valida,grupos,estabelecimentos
versao,,,,,
SUS,92782,90985,45770048,245,245
Todos,92782,90985,45770048,273,274


,setores_com_unidade_diferente,setores_com_reducao_maior_1m,moradores_em_setores_com_reducao_maior_1m,reducao_maxima_m
versao,,,,
SUS,402,389,179670,331.61
Todos,417,405,194163,5125.60


## 6. O que a conferência demonstra

A distância nova é calculada diretamente com o Dijkstra e a regra escolhida: menor soma de ligações e percurso na rua. Os arquivos anteriores são usados somente como comparação.

Se uma unidade diferente vencer, isso mostra que o custo da ligação até a rua alterava a atribuição. Se a distância diminuir, essa redução mede o efeito da nova regra naquele setor; não é melhoria física na cidade nem mudança real no serviço. A comparação não avalia localização ótima, capacidade, tempo de espera ou qualidade do atendimento.

In [8]:
for versao in ["SUS","Todos"]:
    cadastro = unidades.loc[unidades.atende_sus] if versao=="SUS" else unidades
    gs = grupos.loc[grupos.versao==versao]
    codigos = [c for grupo in gs.codigos_unidades for c in grupo.split(";")]
    assert len(codigos)==len(set(codigos))
    assert set(codigos)==set(cadastro.CO_CNES)
assert len(list(DIST.glob("*_setores.csv")))==52
assert comparacoes.setores_validos.sum()>0
print("Conferências concluídas: setores únicos, atributos preservados, regra da menor soma e 52 arquivos gerados.")

Conferências concluídas: setores únicos, atributos preservados, regra da menor soma e 52 arquivos gerados.
